In [11]:
import torch
from Transformer import Transformer
from utils_Transformer import params
import pickle
import io
from traffic_data_loader import data_loader_full, tensor_reshape, Traffic_Flow_Data
import numpy as np

In [12]:
# retreive parameters from params
input_size = params['input_size']
d_model = params['d_model']
num_heads = params['num_heads']
num_encoder_layers = params['num_encoder_layers']
dim_feedforward = params['dim_feedforward']
dropout = params['dropout']
output_size = params['output_size']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [13]:
# load model
model_TE = Transformer(input_size, d_model, num_heads, num_encoder_layers, dim_feedforward, dropout, output_size).to(device)
model_TE.load_state_dict(torch.load('saved_model/model_Transformer.pth', map_location=device))
model_TE.to(device)
model_TE.eval() # enable evaluation mode

Transformer(
  (input_linear): Linear(in_features=17, out_features=128, bias=True)
  (positional_encoding): PositionalEncoding()
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=128, out_features=17, bias=True)
)

In [14]:
# start prediction
class CPU_Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
        else:
            return super().find_class(module, name)


file_path = '../Model_Final/Predicted/results.pkl'
with open(file_path, 'rb') as pickle_file:
    pred = CPU_Unpickler(pickle_file).load()
data = pred['flow_recon'].to(device)

data_occupancy_all, data_flow_all, data_speed_all = data_loader_full()
X_occu_all, _ = data_occupancy_all[:, :2], data_occupancy_all[:, 2]
X_occu_all = torch.tensor(X_occu_all, dtype=torch.float32).to(device)

data = torch.cat((X_occu_all, data), dim=1).detach()

data = tensor_reshape(data)

data_validation = data[int(data.size(0) * 0.6):, :]
# data_validation = data

data_val = Traffic_Flow_Data(data_validation, window_size = params['window_size'])

In [15]:
# define a function to predict n steps
def predict_n_steps(model, data_val, n_steps):
    """
    model: trained LSTM model
    data_val: Traffic_Flow_Data object (validation dataset)
    n_steps: number of recursive prediction steps
    """
    model.eval()
    device = next(model.parameters()).device

    Y_VAL = []
    Y_PRED = []

    with torch.no_grad():
        for idx in range(len(data_val) - n_steps):
            x_val, _ = data_val[idx]
            x_val = x_val.unsqueeze(0).to(device)

            # Recursive prediction
            for step in range(n_steps):
                y_pred = model(x_val)
                y_pred_expand = y_pred.unsqueeze(0)  # (1, 1, feature_dim)

                # Update input for next step
                x_val = torch.cat((x_val, y_pred_expand), dim=1)
                x_val = x_val[:, 1:, :]  # remove the oldest time step

            # After n_steps prediction, collect y_pred and corresponding ground truth
            y_pred = y_pred.cpu().detach().numpy()
            y_pred = np.squeeze(y_pred, axis=0)

            _, y_val = data_val[idx + n_steps]
            y_val = y_val.cpu().detach().numpy()

            Y_VAL.append(y_val)
            Y_PRED.append(y_pred)

    Y_VAL = np.vstack(Y_VAL)
    Y_PRED = np.vstack(Y_PRED)

    return Y_VAL, Y_PRED

In [16]:
n_steps = 1
Y_VAL_1, Y_PRED_1 = predict_n_steps(model_TE, data_val, n_steps)

rmse_1 = np.sqrt(np.nanmean((Y_VAL_1 - Y_PRED_1) ** 2))
mape_1 = np.nanmean(np.abs((Y_VAL_1 - Y_PRED_1) / Y_VAL_1)) * 100

print(f'RMSE ({n_steps}-step): {rmse_1:.4f}')
print(f'MAPE ({n_steps}-step): {mape_1:.2f}%')

RMSE (1-step): 10.6287
MAPE (1-step): 12.38%


In [17]:
n_steps = 2
Y_VAL_2, Y_PRED_2 = predict_n_steps(model_TE, data_val, n_steps)

rmse_2 = np.sqrt(np.nanmean((Y_VAL_2 - Y_PRED_2) ** 2))
mape_2 = np.nanmean(np.abs((Y_VAL_2 - Y_PRED_2) / Y_VAL_2)) * 100

print(f'RMSE ({n_steps}-step): {rmse_2:.4f}')
print(f'MAPE ({n_steps}-step): {mape_2:.2f}%')

RMSE (2-step): 11.0263
MAPE (2-step): 13.46%


In [18]:
n_steps = 3
Y_VAL_3, Y_PRED_3 = predict_n_steps(model_TE, data_val, n_steps)

rmse_3 = np.sqrt(np.nanmean((Y_VAL_3 - Y_PRED_3) ** 2))
mape_3 = np.nanmean(np.abs((Y_VAL_3 - Y_PRED_3) / Y_VAL_3)) * 100

print(f'RMSE ({n_steps}-step): {rmse_3:.4f}')
print(f'MAPE ({n_steps}-step): {mape_3:.2f}%')

RMSE (3-step): 10.9702
MAPE (3-step): 13.46%


In [19]:
n_steps = 4
Y_VAL_4, Y_PRED_4 = predict_n_steps(model_TE, data_val, n_steps)

rmse_4 = np.sqrt(np.nanmean((Y_VAL_4 - Y_PRED_4) ** 2))
mape_4 = np.nanmean(np.abs((Y_VAL_4 - Y_PRED_4) / Y_VAL_4)) * 100

print(f'RMSE ({n_steps}-step): {rmse_4:.4f}')
print(f'MAPE ({n_steps}-step): {mape_4:.2f}%')

RMSE (4-step): 10.9260
MAPE (4-step): 13.41%


In [20]:
n_steps = 5
Y_VAL_5, Y_PRED_5 = predict_n_steps(model_TE, data_val, n_steps)

rmse_5 = np.sqrt(np.nanmean((Y_VAL_5 - Y_PRED_5) ** 2))
mape_5 = np.nanmean(np.abs((Y_VAL_5 - Y_PRED_5) / Y_VAL_5)) * 100

print(f'RMSE ({n_steps}-step): {rmse_5:.4f}')
print(f'MAPE ({n_steps}-step): {mape_5:.2f}%')

RMSE (5-step): 10.8527
MAPE (5-step): 13.34%


## 3 minutes prediction

In [5]:
# create empty np array
Y_VAL_1 = []
Y_PRED_1 = []

In [6]:
for idx in range(data_val.__len__()):
    x_val, y_val = data_val.__getitem__(idx)
    x_val = x_val.unsqueeze(0).to(device)
    y_pred = model_TE(x_val).detach().to('cpu').numpy()
    y_val = y_val.detach().to('cpu').numpy()
    Y_VAL_1.append(y_val)
    Y_PRED_1.append(y_pred)

In [7]:
Y_VAL_1 = np.vstack(Y_VAL_1)
Y_PRED_1 = np.vstack(Y_PRED_1)

In [8]:
# calculate the RMSE and MAPE
rmse_flow_1_TE = np.sqrt(np.nanmean((Y_VAL_1 - Y_PRED_1) ** 2))
print(rmse_flow_1_TE)

mape_flow_1_TE = np.nanmean(np.abs((Y_VAL_1 - Y_PRED_1) / Y_VAL_1)) * 100
print(mape_flow_1_TE)

11.444492
14.533337950706482


## 6 minutes prediction

In [9]:
# create empty np array
Y_VAL_2 = []
Y_PRED_2 = []

In [10]:
for idx in range(data_val.__len__()-1):
    x_val, _ = data_val.__getitem__(idx)
    x_val = x_val.unsqueeze(0).to(device)
    y_pred_1  = model_TE(x_val).unsqueeze(0)

    # add y_pred_1 to the bottom of 'x_val' and remove the first row of original 'x_val'
    x_val_1 = torch.cat((x_val, y_pred_1), dim=1)
    x_val_1 = x_val_1[:,1:,:]
    
    # put x_val_1 into model again and get y_pred_2
    y_pred_2 = model_TE(x_val_1).detach().to('cpu').numpy()
    _, y_val_2 = data_val.__getitem__(idx+1)
    
    y_val_2 = y_val_2.detach().to('cpu').numpy()
    
    Y_VAL_2.append(y_val_2)
    Y_PRED_2.append(y_pred_2)

In [11]:
Y_VAL_2 = np.vstack(Y_VAL_2)
Y_PRED_2 = np.vstack(Y_PRED_2)

In [12]:
# calculate the RMSE and MAPE
rmse_flow_2_TE = np.sqrt(np.nanmean((Y_VAL_2 - Y_PRED_2) ** 2))
print(rmse_flow_2_TE)

mape_flow_2_TE = np.nanmean(np.abs((Y_VAL_2 - Y_PRED_2) / Y_VAL_2)) * 100
print(mape_flow_2_TE)

11.932013
15.180811285972595


## 9 minutes prediction

In [13]:
# create empty np array
Y_VAL_3 = []
Y_PRED_3 = []

In [14]:
for idx in range(data_val.__len__()-2):
    x_val, _ = data_val.__getitem__(idx)
    x_val = x_val.unsqueeze(0).to(device)
    y_pred_1  = model_TE(x_val).unsqueeze(0)

    # add y_pred_1 to the bottom of 'x_val' and remove the first row of original 'x_val'
    x_val_1 = torch.cat((x_val, y_pred_1), dim=1)
    x_val_1 = x_val_1[:,1:,:]
    
    # put x_val_1 into model again and get y_pred_2
    y_pred_2 = model_TE(x_val_1).unsqueeze(0)
    
    x_val_2 = torch.cat((x_val_1, y_pred_2), dim=1)
    x_val_2 = x_val_2[:,1:,:]
    
    y_pred_3 = model_TE(x_val_2).detach().to('cpu').numpy()
    
    _, y_val_3 = data_val.__getitem__(idx+2)
    
    y_val_3 = y_val_3.detach().to('cpu').numpy()
    
    Y_VAL_3.append(y_val_3)
    Y_PRED_3.append(y_pred_3)

In [15]:
Y_VAL_3 = np.vstack(Y_VAL_3)
Y_PRED_3 = np.vstack(Y_PRED_3)

In [16]:
rmse_flow_3_TE = np.sqrt(np.nanmean((Y_VAL_3 - Y_PRED_3) ** 2))
print(rmse_flow_3_TE)

mape_flow_3_TE = np.nanmean(np.abs((Y_VAL_3 - Y_PRED_3) / Y_VAL_3)) * 100
print(mape_flow_3_TE)

12.503949
15.799927711486816


## 12 minutes prediction

In [17]:
# create empty np array
Y_VAL_4 = []
Y_PRED_4 = []

In [18]:
for idx in range(data_val.__len__()-3):
    x_val, _ = data_val.__getitem__(idx)
    x_val = x_val.unsqueeze(0).to(device)
    y_pred_1  = model_TE(x_val).unsqueeze(0)

    # add y_pred_1 to the bottom of 'x_val' and remove the first row of original 'x_val'
    x_val_1 = torch.cat((x_val, y_pred_1), dim=1)
    x_val_1 = x_val_1[:,1:,:]
    
    # put x_val_1 into model again and get y_pred_2
    y_pred_2 = model_TE(x_val_1).unsqueeze(0)
    
    x_val_2 = torch.cat((x_val_1, y_pred_2), dim=1)
    x_val_2 = x_val_2[:,1:,:]
    
    y_pred_3 = model_TE(x_val_2).unsqueeze(0)
    
    x_val_3 = torch.cat((x_val_2, y_pred_3), dim=1)
    x_val_3 = x_val_3[:,1:,:]
    
    y_pred_4 = model_TE(x_val_3).detach().to('cpu').numpy()
    
    _, y_val_4 = data_val.__getitem__(idx+3)
    
    y_val_4 = y_val_4.detach().to('cpu').numpy()
    
    Y_VAL_4.append(y_val_4)
    Y_PRED_4.append(y_pred_4)

In [19]:
Y_VAL_4 = np.vstack(Y_VAL_4)
Y_PRED_4 = np.vstack(Y_PRED_4)

In [20]:
rmse_flow_4_TE = np.sqrt(np.nanmean((Y_VAL_4 - Y_PRED_4) ** 2))
print(rmse_flow_4_TE)

mape_flow_4_TE = np.nanmean(np.abs((Y_VAL_4 - Y_PRED_4) / Y_VAL_4)) * 100
print(mape_flow_4_TE)

12.999815
16.35783314704895


## 15 minutes prediction

In [21]:
# create empty np array
Y_VAL_5 = []
Y_PRED_5 = []

In [22]:
for idx in range(data_val.__len__()-4):
    x_val, _ = data_val.__getitem__(idx)
    x_val = x_val.unsqueeze(0).to(device)
    y_pred_1  = model_TE(x_val).unsqueeze(0)

    # add y_pred_1 to the bottom of 'x_val' and remove the first row of original 'x_val'
    x_val_1 = torch.cat((x_val, y_pred_1), dim=1)
    x_val_1 = x_val_1[:,1:,:]
    
    # put x_val_1 into model again and get y_pred_2
    y_pred_2 = model_TE(x_val_1).unsqueeze(0)
    
    x_val_2 = torch.cat((x_val_1, y_pred_2), dim=1)
    x_val_2 = x_val_2[:,1:,:]
    
    y_pred_3 = model_TE(x_val_2).unsqueeze(0)
    
    x_val_3 = torch.cat((x_val_2, y_pred_3), dim=1)
    x_val_3 = x_val_3[:,1:,:]
    
    y_pred_4 = model_TE(x_val_3).unsqueeze(0)
    
    x_val_4 = torch.cat((x_val_3, y_pred_4), dim=1)
    x_val_4 = x_val_4[:,1:,:]
    
    y_pred_5 = model_TE(x_val_4).detach().to('cpu').numpy()
    
    _, y_val_5 = data_val.__getitem__(idx+4)
    
    y_val_5 = y_val_5.detach().to('cpu').numpy()
    
    Y_VAL_5.append(y_val_5)
    Y_PRED_5.append(y_pred_5)

In [23]:
Y_VAL_5 = np.vstack(Y_VAL_5)
Y_PRED_5 = np.vstack(Y_PRED_5)

In [24]:
rmse_flow_5_TE = np.sqrt(np.nanmean((Y_VAL_5 - Y_PRED_5) ** 2))
print(rmse_flow_5_TE)

mape_flow_5_TE = np.nanmean(np.abs((Y_VAL_5 - Y_PRED_5) / Y_VAL_5)) * 100
print(mape_flow_5_TE)

13.360388
16.791580617427826


In [25]:
import pandas as pd

In [26]:
data = {
    'Prediction Length': ['RMSE(%)', 'MAPE(%)'],
    '3-min prediction': [rmse_flow_1_TE, mape_flow_1_TE],
    '6-min prediction': [rmse_flow_2_TE, mape_flow_2_TE],
    '9-min prediction': [rmse_flow_3_TE, mape_flow_3_TE],
    '12-min prediction': [rmse_flow_4_TE, mape_flow_4_TE],
    '15-min prediction': [rmse_flow_5_TE, mape_flow_5_TE]
}
df = pd.DataFrame(data)
df.iloc[:, 1:] = df.iloc[:, 1:].round(2)
df.to_csv('Tables/Prediction_Error_TE.csv', index=False)